# 13. Paoay maps for every fixed-window start month

One fixed-window model per start month (Jan–Apr … Sep–Dec), trained on the development parcels (train + validation) with the pipeline's baseline settings, as in notebook 06. Each model is applied to Paoay pixels whose eight half-months are all clear in the local Cloud Score+ composites, for the year with the most such cropland pixels. Unsure pixels become Unknown (0.60 / 0.10 rule).

No test parcels are used; frozen models, the main Paoay outputs and stage pointers are unchanged. Imagery differs from training as in Paoay notebook 07. Exploratory: no local labels.

**Input:** Preprocessing 01 quality pointer, `atok_imagery_root`, land-activity map. **Output:** `pipeline_outputs/transfer/start_month_maps_*`.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # model_aza

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display
from model_aza.training import start_month_maps as maps, start_month_sweep as sweep, transfer, tuning
from crop_pipeline.paoay import aligned
pd.set_option("display.width", 200)

## Train one model per start month

In [ ]:
quality_path = ""  # Optional quality.json; blank uses the verified Preprocessing 01 pointer
months = list(sweep.START_MONTHS)
params = dict(tuning.BASELINE)
quality_handoff, inputs = sweep.load_quality(root, quality_path or None)
models = maps.fit_models(inputs, months, params, progress=lambda m: print(m, flush=True))
del inputs
run = tuning.new_run(root, "start_month_maps", quality_handoff, area=transfer.AREA, start_months=months,
                     params={k: str(v) for k, v in params.items()}, years=list(maps.YEARS))
print("Run:", run)

## Paoay study area and cropland

In [ ]:
from model_aza.improved import workflow
paoay = workflow.paoay_inputs(root)
settings = json.loads((root / "settings.local.json").read_text())
imagery_root = Path(settings["atok_imagery_root"]).expanduser()
imagery_root = imagery_root if imagery_root.is_absolute() else (root / imagery_root).resolve()
grid, mask, pixels = paoay["grid"], paoay["mask"], paoay["pixels"]
cropland = np.zeros(mask.shape, bool)
cropland[tuple(pixels[paoay["cropland"]].T)] = True
print("Study pixels:", int(mask.sum()), "| land-activity cropland pixels:", int(cropland.sum()))

## Clear four-month windows in Paoay cropland, by year

In [ ]:
cubes, coverage = {}, []
for year in maps.YEARS:
    cubes[year], _ = aligned.read_year(imagery_root, year, grid)
    for month in models:
        clear = maps.complete(cubes[year], month)
        coverage.append({"year": year, "start_month": month, "window": maps.label(month),
                         "cropland_complete_share": float(clear[cropland].mean())})
tuning.save_records(run, "coverage", coverage)
display(pd.DataFrame(coverage).pivot_table(index="window", columns="year", values="cropland_complete_share", sort=False).round(3))

## Classify each window in its clearest year

In [ ]:
summary, rasters = [], {}
for month, (model, development_pixels) in models.items():
    best = max((r for r in coverage if r["start_month"] == month), key=lambda r: (r["cropland_complete_share"], r["year"]))
    year, cube = best["year"], cubes[best["year"]]
    clear = maps.complete(cube, month) & mask
    labels = maps.classify(model, cube, year, month, grid, np.argwhere(clear))
    rasters[month] = maps.code_raster(mask, labels)
    summary.append({**maps.summarize(month, year, labels, mask, cropland, best["cropland_complete_share"]),
                    "development_pixels": development_pixels})
    print(f"{maps.label(month)} {year}: {len(labels)} classified pixels", flush=True)
del cubes
tuning.save_records(run, "summary", summary)
np.savez_compressed(run / "maps.npz", **{maps.label(m): r for m, r in rasters.items()})
statistics = pd.read_csv(root / "data/paoay/benguet_vegetable_area_2024.csv")
psa = (statistics.set_index("crop")["area_ha"] / statistics["area_ha"].sum()).to_dict()
table = pd.DataFrame(summary).set_index("window")
display(table[["year", "cropland_complete_share", "cropland_confident_pixels",
               "carrot_share_of_confident", "cabbage_share_of_confident", "potato_share_of_confident"]].round(3))
print("PSA Benguet 2024 shares:", {k: round(v, 3) for k, v in psa.items()})

## Development scores per start month (from notebook 06)

In [ ]:
sweep_run = tuning.latest_run(root, "start_month_sweep", area=transfer.AREA)
scores = pd.read_csv(sweep_run / "month_scores.csv")
scores = scores[scores["variant"] == "interval_72"].assign(window=lambda d: d.start_month.map(maps.label)).set_index("window")
display(scores[["spatial_macro_f1", "locked_validation_macro_f1", "source_holdout_macro_f1_mean"]].round(3))
print("Sweep run:", sweep_run)

## Maps

In [ ]:
colors = ListedColormap(["#ffffff", "#ff8c00", "#9acd60", "#7a5444", "#a0a0a0", "#a8c8ec"])
figure, axes = plt.subplots(3, 3, figsize=(15, 13))
for axis, (month, codes) in zip(axes.flat, rasters.items()):
    row = next(s for s in summary if s["start_month"] == month)
    axis.imshow(codes, cmap=colors, vmin=0, vmax=5, interpolation="nearest")
    axis.set_title(f"{maps.label(month)} {row['year']}: {row['cropland_complete_share']:.0%} of cropland clear", fontsize=10)
    axis.axis("off")
for axis in axes.flat[len(rasters):]:
    axis.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=colors(i)) for i in range(1, 6)]
figure.legend(handles, ["Carrot", "Cabbage", "Potato", "Unknown crop type", "Window not cloud-free"],
              loc="lower center", ncol=5, frameon=False)
figure.suptitle("Fixed-window models by start month, Paoay (exploratory, not validated)")
figure.tight_layout(rect=(0, .04, 1, .97))
figure.savefig(run / "start_month_maps.png", dpi=120)
display(figure)
plt.close(figure)

## Crop shares of confident cropland

In [ ]:
shares = table[["carrot_share_of_confident", "cabbage_share_of_confident", "potato_share_of_confident"]].fillna(0)
shares.columns = ["carrot", "cabbage", "potato"]
shares.loc["PSA Benguet 2024"] = [psa["carrot"], psa["cabbage"], psa["potato"]]
figure, axis = plt.subplots(figsize=(11, 4))
shares.plot.bar(stacked=True, ax=axis, color=["#ff8c00", "#9acd60", "#7a5444"], width=.75)
for i, window in enumerate(table.index):
    axis.text(i, 1.02, f"{int(table.loc[window, 'cropland_confident_pixels'])} px", ha="center", fontsize=8)
axis.set_ylim(0, 1.12)
axis.set_ylabel("Share of confident cropland pixels")
axis.set_xlabel("")
axis.tick_params(axis="x", rotation=0, labelsize=8)
axis.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
figure.tight_layout()
figure.savefig(run / "crop_shares.png", dpi=150)
display(figure)
plt.close(figure)